In [ ]:
%matplotlib inline
import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.family'] = 'monospace'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['figure.facecolor'] = '#fafafa'
mpl.rcParams['axes.facecolor'] = '#fafafa'

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(2021)

n_weeks = 156
dates = pd.date_range(start='2021-01-04', periods=n_weeks, freq='W-MON')

# --- IBEX 35 price series (random walk, range 8200-9800) ---
price_start = 9000
steps = np.random.normal(0, 50, n_weeks)
prices_raw = price_start + np.cumsum(steps)
# rescale to [8200, 9800]
prices_min = prices_raw.min()
prices_max = prices_raw.max()
ibex_prices = 8200 + (prices_raw - prices_min) / (prices_max - prices_min) * 1600

# --- Weekly returns ---
ibex_returns = np.diff(ibex_prices) / ibex_prices[:-1]
ibex_returns = np.append(ibex_returns, np.nan)

# --- Sentiment scores (-1 to 1) ---
# lag-1 correlation injected by construction: sentiment at t encodes next week's return direction
noise = np.random.normal(0, 0.3, n_weeks)
signal = np.roll(ibex_returns, 1)  # sentiment[t] correlates with return[t+1]
signal[0] = 0
signal[-1] = 0
sentiment_raw = 0.6 * signal / (np.nanstd(signal) + 1e-9) + noise
# clip to [-1, 1]
sentiment = np.clip(sentiment_raw, -1, 1)

df = pd.DataFrame({
    'date': dates,
    'ibex_price': ibex_prices,
    'ibex_return': ibex_returns,
    'sentiment': sentiment
})

print('Shape:', df.shape)
print(df.head())

In [ ]:
# --- Sentiment distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribucion del sentimiento semanal (2021-2024)', fontsize=13, y=1.01)

# Histogram
ax1 = axes[0]
ax1.hist(df['sentiment'], bins=30, color='#4c72b0', edgecolor='white', linewidth=0.5)
ax1.axvline(0, color='#c44e52', linestyle='--', linewidth=1.2, label='neutro')
ax1.set_xlabel('Sentiment score')
ax1.set_ylabel('Frecuencia')
ax1.set_title('Histograma')
ax1.legend(fontsize=9)

# Positive vs negative weeks
ax2 = axes[1]
n_pos = (df['sentiment'] > 0).sum()
n_neg = (df['sentiment'] <= 0).sum()
bars = ax2.bar(['Positivas', 'Negativas / neutras'], [n_pos, n_neg],
               color=['#55a868', '#c44e52'], edgecolor='white')
for bar, val in zip(bars, [n_pos, n_neg]):
    label_str = str(val)
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             label_str, ha='center', va='bottom', fontsize=10)
ax2.set_ylabel('Numero de semanas')
ax2.set_title('Semanas positivas vs negativas')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import correlate  # will use for cross-correlogram later

# --- Dual-axis time series: IBEX price vs sentiment ---
fig, ax1 = plt.subplots(figsize=(14, 5))

color_ibex = '#2c5f8a'
color_sent = '#c44e52'

ax1.plot(df['date'], df['ibex_price'], color=color_ibex, linewidth=1.5, label='IBEX 35 (precio)')
ax1.set_xlabel('Fecha')
ax1.set_ylabel('IBEX 35 (puntos)', color=color_ibex)
ax1.tick_params(axis='y', labelcolor=color_ibex)
ax1.set_title('IBEX 35 vs Sentimiento de titulares economicos (2021-2024)', fontsize=12)

ax2 = ax1.twinx()
ax2.fill_between(df['date'], df['sentiment'], 0,
                 where=(df['sentiment'] >= 0), alpha=0.4, color='#55a868', label='sentimiento positivo')
ax2.fill_between(df['date'], df['sentiment'], 0,
                 where=(df['sentiment'] < 0), alpha=0.4, color=color_sent, label='sentimiento negativo')
ax2.plot(df['date'], df['sentiment'], color='#333333', linewidth=0.6, alpha=0.5)
ax2.set_ylabel('Sentiment score', color='#333333')
ax2.tick_params(axis='y', labelcolor='#333333')
ax2.set_ylim(-1.5, 1.5)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()